# 03 — Customer SCD Type 2
Demonstrates historical tracking for customer attributes. The Olist source is a static historical extract, not a CDC stream. Therefore this notebook seeds history from the snapshot and applies one deterministic simulated address change to demonstrate the Type 2 expire-and-insert pattern.

**Run note:** This notebook intentionally rebuilds the demo history table from the Silver snapshot each time, then applies the same simulated change. This keeps the demonstration repeatable. In a real incremental pipeline, do not reset history; process genuine source changes using source effective timestamps or CDC events.

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

tracked = ["customer_unique_id", "customer_zip_code_prefix", "customer_city", "customer_state"]

def with_hash(df):
    return df.withColumn(
        "record_hash",
        F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in tracked]), 256)
    )

# Seed one current version per customer from the Silver snapshot.
base = with_hash(spark.table("silver.customers"))
history = (base
    .select("customer_id", *tracked, "record_hash")
    .withColumn("effective_start_ts", F.current_timestamp())
    .withColumn("effective_end_ts", F.to_timestamp(F.lit("9999-12-31 23:59:59")))
    .withColumn("is_current", F.lit(True)))

(history.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable("silver.customer_history"))

print("Seeded SCD Type 2 history:", history.count())

## Simulate one customer address change
The first customer is selected deterministically. We modify its city with a clearly labelled demo suffix. This is a test change, not a claim that the source contained a real update.

In [ ]:
# Deterministic demonstration update
sample = (spark.table("silver.customers")
          .orderBy("customer_id")
          .limit(1)
          .withColumn("customer_city", F.concat(F.col("customer_city"), F.lit("_demo_updated"))))

incoming = with_hash(sample).withColumn("effective_ts", F.current_timestamp())

# Expire the current row only when the tracked attributes have changed.
target = DeltaTable.forName(spark, "silver.customer_history")
(target.alias("t")
 .merge(incoming.alias("s"), "t.customer_id = s.customer_id AND t.is_current = true")
 .whenMatchedUpdate(
     condition="t.record_hash <> s.record_hash",
     set={
         "is_current": "false",
         "effective_end_ts": "s.effective_ts"
     }
 ).execute())

# Insert a new current version if there is no current row with the incoming hash.
current = (spark.table("silver.customer_history")
           .filter(F.col("is_current") == True)
           .select("customer_id", "record_hash"))

new_versions = (incoming.alias("s")
    .join(current.alias("c"),
          (F.col("s.customer_id") == F.col("c.customer_id")) &
          (F.col("s.record_hash") == F.col("c.record_hash")),
          "left_anti")
    .select("s.customer_id", *[F.col(f"s.{c}").alias(c) for c in tracked],
            F.col("s.record_hash"),
            F.col("s.effective_ts").alias("effective_start_ts"))
    .withColumn("effective_end_ts", F.to_timestamp(F.lit("9999-12-31 23:59:59")))
    .withColumn("is_current", F.lit(True)))

(new_versions.write.format("delta").mode("append").saveAsTable("silver.customer_history"))

display(spark.table("silver.customer_history")
        .filter(F.col("customer_id") == sample.first()["customer_id"])
        .orderBy("effective_start_ts"))

## Validate SCD Type 2 invariants
Each customer must have no more than one current row. The changed demo customer should have two versions.

In [ ]:
multiple_current = (spark.table("silver.customer_history")
    .filter("is_current = true")
    .groupBy("customer_id").count()
    .filter("count > 1"))

assert multiple_current.count() == 0, "More than one current version found for a customer"

demo_customer_id = sample.first()["customer_id"]
demo_versions = spark.table("silver.customer_history").filter(F.col("customer_id") == demo_customer_id)
assert demo_versions.count() == 2, "Expected two historical versions for simulated customer"

print("SCD Type 2 validation passed for demo customer:", demo_customer_id)